# Energy: Turbine Performance Trending

Load multiple sessions recorded over time and track turbine efficiency metrics to identify performance degradation trends.

**Context:** In power generation, turbine performance is monitored over weeks or months. Each test session captures temperature, pressure, and rotational speed. Trending these metrics reveals gradual degradation that warrants maintenance.

In [ ]:
import sys
sys.path.insert(0, '..')
from sqlrace_helpers import (
    init_sqlrace, load_session, session_summary,
    list_parameters, extract_parameters
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")

# Session GUIDs and their .ssn2 files in chronological order
SESSIONS = [
    ("e6d9bbc0-89a6-4754-9b4c-f7b5a6b4718b", "turbine-week1.ssn2"),  # Week 1
    ("7e74ba09-b54d-415e-bb25-4214300019ef", "turbine-week2.ssn2"),  # Week 2
    ("56d725d0-569b-4c74-b6ae-4dd497edca2d", "turbine-week3.ssn2"),  # Week 3
]

# Parameters to trend
TREND_PARAMS = ["Temperature:Sensors", "Pressure:Inlet", "RPM:Turbine"]

sm = init_sqlrace()

## Load sessions and compute per-session statistics

For each session, extract the target parameters and compute summary statistics.

In [ ]:
session_stats = []

for idx, (guid, filename) in enumerate(SESSIONS):
    cs = f"DbEngine=SQLite;Data Source={os.path.join(DATA_DIR, filename)};"
    client, sess = load_session(sm, guid, connection_string=cs)
    try:
        available = set(list_parameters(sess))
        params = [p for p in TREND_PARAMS if p in available]

        df = extract_parameters(sess, params)
        row = {"Session": idx + 1, "GUID": guid[:8] + "..."}

        for col in df.columns:
            row[f"{col}_mean"] = df[col].mean()
            row[f"{col}_std"] = df[col].std()

        session_stats.append(row)
        print(f"  Session {idx + 1}: {len(df)} samples, {len(params)} params")
    finally:
        client.Dispose()

df_trend = pd.DataFrame(session_stats).set_index("Session")
display(df_trend)

## Trend plots

Plot mean values over time to visualise degradation.

In [ ]:
mean_cols = [c for c in df_trend.columns if c.endswith("_mean")]
n = len(mean_cols)

fig, axes = plt.subplots(n, 1, figsize=(10, 3 * n), sharex=True, squeeze=False)

for i, col in enumerate(mean_cols):
    param_name = col.replace("_mean", "")
    std_col = f"{param_name}_std"

    ax = axes[i, 0]
    x = df_trend.index
    y = df_trend[col]

    ax.plot(x, y, 'o-', markersize=6, linewidth=1.5)

    if std_col in df_trend.columns:
        err = df_trend[std_col]
        ax.fill_between(x, y - err, y + err, alpha=0.2)

    # Add trend line
    z = np.polyfit(x, y, 1)
    ax.plot(x, np.polyval(z, x), '--', color='red', alpha=0.6,
            label=f"Trend: {z[0]:+.4f}/session")

    ax.set_ylabel(param_name)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0, 0].set_title("Turbine Performance Trending")
axes[-1, 0].set_xlabel("Session number")
plt.tight_layout()
plt.show()

## Degradation assessment

Flag parameters where the trend slope exceeds a threshold.

In [ ]:
THRESHOLD_PCT = 5.0  # Flag if >5% change over the series

print("Degradation Assessment")
print("=" * 60)

for col in mean_cols:
    param_name = col.replace("_mean", "")
    values = df_trend[col].values
    if len(values) < 2 or values[0] == 0:
        continue

    pct_change = 100 * (values[-1] - values[0]) / abs(values[0])
    status = "WARNING" if abs(pct_change) > THRESHOLD_PCT else "OK"
    print(f"  {param_name:30s}  {pct_change:+.2f}%  [{status}]")